In [ ]:
# Cell 1: Install all required dependencies

# CRITICAL: Upgrade huggingface_hub FIRST
!pip install -q --upgrade huggingface_hub>=0.20.0

# Upgrade transformers and accelerate
!pip install -q --upgrade transformers>=4.35.0
!pip install -q --upgrade accelerate>=0.25.0

# Install other dependencies
!pip install -q scikit-learn imbalanced-learn sentencepiece

print("✓ All dependencies installed successfully!")

# Verify versions
import transformers
import huggingface_hub
import torch

print(f"\n📦 Versions:")
print(f"  transformers: {transformers.__version__}")
print(f"  huggingface_hub: {huggingface_hub.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")


In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import Counter
import re
from typing import List, Dict, Tuple

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Transformers (CORRECTED)
# Transformers (✅ CORRECTED)
from transformers import (
    XLMRobertaTokenizer, 
    XLMRobertaForSequenceClassification,
    get_scheduler  # ✅ This fixes the scheduler import error
)


# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    confusion_matrix, 
    classification_report,
    roc_auc_score,
    roc_curve,
    f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

# Imbalanced-learn
from imblearn.over_sampling import SMOTE

# Suppress warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✓ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 3: Set random seeds
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility across all frameworks."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✓ Random seed set to {seed} for reproducibility")

set_seed(42)

# Configuration
CONFIG = {
    'seed': 42,
    'max_length': 512,
    'batch_size': 16,
    'learning_rate': 2e-5,
    'epochs': 10,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'early_stopping_patience': 3,
    'model_name': 'xlm-roberta-base',
    'test_size': 0.2,
    'val_size': 0.25,  # 0.25 of 0.8 = 0.2 overall
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print("\n📝 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


In [ ]:
# Cell 4: Load datasets with UTF-8 encoding
def load_data(true_path: str, false_path: str) -> pd.DataFrame:
    """
    Load true and false news datasets and combine them.
    
    Args:
        true_path: Path to true_news.csv
        false_path: Path to false_news.csv
    
    Returns:
        Combined DataFrame with proper encoding
    """
    print("📂 Loading datasets...")
    
    # Load datasets with UTF-8 encoding
    df_true = pd.read_csv(true_path, encoding='utf-8')
    df_false = pd.read_csv(false_path, encoding='utf-8')
    
    print(f"  True news: {len(df_true):,} rows")
    print(f"  False news: {len(df_false):,} rows")
    
    # Combine datasets
    df = pd.concat([df_true, df_false], ignore_index=True)
    print(f"  Combined: {len(df):,} rows")
    
    return df

# Update these paths to your actual file locations
TRUE_NEWS_PATH = '/kaggle/input/b-dataset/true_news.csv'
FALSE_NEWS_PATH = '/kaggle/input/b-dataset/false_news.csv'

# Load data
df = load_data(TRUE_NEWS_PATH, FALSE_NEWS_PATH)

print("\n✓ Data loaded successfully!")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")


In [ ]:
# Cell 5: Check data integrity
def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate data integrity and handle missing values.
    
    Args:
        df: Input DataFrame
    
    Returns:
        Cleaned DataFrame
    """
    print("🔍 Data Validation Report")
    print("=" * 60)
    
    # Display basic info
    print(f"\n1. Dataset Shape: {df.shape}")
    print(f"   Rows: {df.shape[0]:,}")
    print(f"   Columns: {df.shape[1]}")

    
    # Check data types
    print(f"\n2. Data Types:")
    print(df.dtypes)
    
    # Check missing values
    print(f"\n3. Missing Values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    })
    print(missing_df[missing_df['Missing Count'] > 0])
    
    # Handle missing values in critical columns
    critical_cols = ['headline', 'descriptions', 'label']
    
    for col in critical_cols:
        if col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f"\n  ⚠️  Found {missing_count} missing values in '{col}'")
                if col in ['headline', 'descriptions']:
                    df[col].fillna('', inplace=True)
                    print(f"     → Filled with empty string")
    
    # Check for duplicate rows
    duplicates = df.duplicated().sum()
    print(f"\n4. Duplicate Rows: {duplicates}")
    if duplicates > 0:
        df = df.drop_duplicates()
        print(f"   → Removed {duplicates} duplicate rows")
    
    # Verify label column
    if 'label' in df.columns:
        print(f"\n5. Label Distribution:")
        print(df['label'].value_counts())
    
    print("\n" + "=" * 60)
    print("✓ Validation complete!")
    
    return df

df = validate_data(df)

# Display sample rows
print("\n📊 Sample Rows (First 3):")
print(df.head(3))

print("\n📊 Sample Rows (Random 3):")
print(df.sample(3, random_state=42))


In [ ]:
# Cell 6: Generate descriptive statistics
def generate_statistics(df: pd.DataFrame):
    """Generate and display descriptive statistics."""
    print("📈 Descriptive Statistics")
    print("=" * 60)
    
    # Numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\nNumeric Features:")
        print(df[numeric_cols].describe())
    
    # Text length statistics
    if 'headline' in df.columns:
        df['headline_length'] = df['headline'].fillna('').str.len()
    if 'descriptions' in df.columns:
        df['desc_length'] = df['descriptions'].fillna('').str.len()
    
    if 'headline_length' in df.columns and 'desc_length' in df.columns:
        print("\n\nText Length Statistics:")
        print(df[['headline_length', 'desc_length']].describe())
    
    # Categorical columns
    categorical_cols = df.select_dtypes(include=['object']).columns
    print(f"\n\nCategorical Features: {categorical_cols.tolist()}")
    
    for col in ['source', 'category', 'label']:
        if col in df.columns:
            print(f"\n{col.upper()} - Unique values: {df[col].nunique()}")
            print(df[col].value_counts().head(10))
    
    print("\n" + "=" * 60)

generate_statistics(df)


In [ ]:
# Cell 7: Visualize class distribution
def plot_class_distribution(df: pd.DataFrame, label_col: str = 'label'):
    """Plot class distribution with detailed statistics."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Count plot
    class_counts = df[label_col].value_counts()
    ax1 = axes[0]
    class_counts.plot(kind='bar', ax=ax1, color=sns.color_palette("Set2"))
    ax1.set_title('Class Distribution (Counts)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Class Label', fontsize=12)
    ax1.set_ylabel('Count', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    
    # Add count labels on bars
    for i, v in enumerate(class_counts.values):
        ax1.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')
    
    # Pie chart
    ax2 = axes[1]
    colors = sns.color_palette("Set2", len(class_counts))
    wedges, texts, autotexts = ax2.pie(
        class_counts.values, 
        labels=class_counts.index,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    ax2.set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print("\n📊 Class Distribution Statistics:")
    print("=" * 60)
    total = len(df)
    for label, count in class_counts.items():
        percentage = (count / total) * 100
        print(f"  {label:20s}: {count:6,} ({percentage:5.2f}%)")
    print("=" * 60)
    
    # Check for imbalance
    max_ratio = class_counts.max() / class_counts.min()
    print(f"\n⚖️  Imbalance Ratio: {max_ratio:.2f}:1")
    if max_ratio > 1.5:
        print("  ⚠️  Dataset is imbalanced - will apply SMOTE or class weights")
    else:
        print("  ✓ Dataset is relatively balanced")

plot_class_distribution(df)


In [ ]:
# Cell 8: Detect and analyze language distribution
def detect_language(text: str) -> str:
    """
    Simple language detector for Bangla vs English.
    
    Args:
        text: Input text string
    
    Returns:
        'bangla', 'english', or 'mixed'
    """
    if pd.isna(text) or text == '':
        return 'unknown'
    
    # Bangla Unicode range: \u0980-\u09FF
    bangla_chars = len(re.findall(r'[\u0980-\u09FF]', text))
    # English letters
    english_chars = len(re.findall(r'[a-zA-Z]', text))
    
    total_alpha = bangla_chars + english_chars
    
    if total_alpha == 0:
        return 'unknown'
    
    bangla_ratio = bangla_chars / total_alpha
    
    if bangla_ratio > 0.8:
        return 'bangla'
    elif bangla_ratio < 0.2:
        return 'english'
    else:
        return 'mixed'

# Apply language detection
print("🌐 Detecting languages...")
df['language'] = df['headline'].fillna('') + ' ' + df['descriptions'].fillna('')
df['language'] = df['language'].apply(detect_language)

# Visualize language distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall language distribution
lang_counts = df['language'].value_counts()
ax1 = axes[0]  # Access first axis
# ax2 = axes[1] 
lang_counts.plot(kind='bar', ax=ax1, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
ax1.set_title('Language Distribution (Overall)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Language', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=0)

for i, v in enumerate(lang_counts.values):
    ax1.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

# Language distribution by class
ax2 = axes[1]
lang_by_class = pd.crosstab(df['label'], df['language'])
lang_by_class.plot(kind='bar', stacked=True, ax=ax2, 
                   color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
ax2.set_title('Language Distribution by Class', fontsize=14, fontweight='bold')
ax2.set_xlabel('Class Label', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Language', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('language_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n🌍 Language Statistics:")
print("=" * 60)
for lang, count in lang_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {lang.capitalize():10s}: {count:6,} ({pct:5.2f}%)")
print("=" * 60)

print("\n📊 Language by Class:")
print(lang_by_class)


In [ ]:
# Cell 9: Analyze text lengths
def analyze_text_lengths(df: pd.DataFrame):
    """Analyze and visualize text length distributions."""
    
    # Create combined text column
    df['combined_text'] = (df['headline'].fillna('') + ' ' + 
                           df['descriptions'].fillna('')).str.strip()
    
    # Calculate word counts
    df['word_count'] = df['combined_text'].apply(lambda x: len(str(x).split()))
    df['char_count'] = df['combined_text'].apply(lambda x: len(str(x)))
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Word count distribution by class
    ax1 = axes[0, 0]
    for label in df['label'].unique():
        subset = df[df['label'] == label]['word_count']
        ax1.hist(subset, bins=50, alpha=0.6, label=label, edgecolor='black')
    ax1.set_title('Word Count Distribution by Class', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Word Count', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Character count distribution by class
    ax2 = axes[0, 1]
    for label in df['label'].unique():
        subset = df[df['label'] == label]['char_count']
        ax2.hist(subset, bins=50, alpha=0.6, label=label, edgecolor='black')
    ax2.set_title('Character Count Distribution by Class', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Character Count', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Box plot - word count by class
    ax3 = axes[1, 0]
    df.boxplot(column='word_count', by='label', ax=ax3, patch_artist=True,
               boxprops=dict(facecolor='lightblue', alpha=0.7))
    ax3.set_title('Word Count by Class (Boxplot)', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Class Label', fontsize=12)
    ax3.set_ylabel('Word Count', fontsize=12)
    plt.sca(ax3)
    plt.xticks(rotation=45)
    
    # Box plot - word count by language
    ax4 = axes[1, 1]
    df.boxplot(column='word_count', by='language', ax=ax4, patch_artist=True,
               boxprops=dict(facecolor='lightgreen', alpha=0.7))
    ax4.set_title('Word Count by Language (Boxplot)', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Language', fontsize=12)
    ax4.set_ylabel('Word Count', fontsize=12)
    plt.sca(ax4)
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig('text_length_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistics
    print("\n📏 Text Length Statistics by Class:")
    print("=" * 80)
    stats = df.groupby('label')[['word_count', 'char_count']].agg(['mean', 'median', 'std', 'min', 'max'])
    print(stats.round(2))
    print("=" * 80)
    
    print("\n📏 Text Length Statistics by Language:")
    print("=" * 80)
    stats_lang = df.groupby('language')[['word_count', 'char_count']].agg(['mean', 'median', 'std'])
    print(stats_lang.round(2))
    print("=" * 80)

analyze_text_lengths(df)


In [ ]:
# Cell 10: Analyze temporal patterns (BULLETPROOF VERSION)

def analyze_temporal_patterns(df: pd.DataFrame, date_col: str = 'date'):
    """Analyze temporal patterns in the dataset."""
    
    if date_col not in df.columns:
        print(f"⚠️  Column '{date_col}' not found. Skipping temporal analysis.")
        return
    
    # Work on a copy
    df_work = df.copy()
    
    # DIAGNOSTIC: Check original data
    print(f"📊 Checking '{date_col}' column:")
    print(f"   Original dtype: {df_work[date_col].dtype}")
    print(f"   Sample values: {df_work[date_col].head(3).tolist()}")
    print(f"   Null count: {df_work[date_col].isna().sum()}/{len(df_work)}")
    
    # Convert to datetime
    df_work[date_col] = pd.to_datetime(df_work[date_col], errors='coerce')
    
    # Count successful conversions
    valid_count = df_work[date_col].notna().sum()
    total_count = len(df_work)
    
    print(f"\n📅 Datetime conversion result: {valid_count}/{total_count} valid dates")
    
    # Exit if no valid dates
    if valid_count == 0:
        print("\n⚠️  NO VALID DATES FOUND - Skipping temporal analysis")
        print("\nPossible reasons:")
        print("  • Date column has incompatible format")
        print("  • All date values are null/missing")
        print("  • Dates need specific format string (e.g., format='%d-%m-%Y')")
        print("\nTemporal analysis will be skipped. Continuing with next cell...")
        return
    
    # Filter for valid dates
    df_valid = df_work[df_work[date_col].notna()].copy()
    
    # CRITICAL CHECK: Verify datetime type
    if not pd.api.types.is_datetime64_any_dtype(df_valid[date_col]):
        print(f"\n⚠️  ERROR: Column '{date_col}' is not datetime type after conversion")
        print(f"   Actual type: {df_valid[date_col].dtype}")
        print("   Temporal analysis cannot proceed. Skipping...")
        return
    
    print(f"✅ Successfully converted {valid_count} dates")
    
    # Extract time features (NOW SAFE)
    df_valid['year'] = df_valid[date_col].dt.year
    df_valid['month'] = df_valid[date_col].dt.month
    df_valid['day_of_week'] = df_valid[date_col].dt.dayofweek
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Articles over time
    ax1 = axes[0, 0]
    try:
        df_valid.groupby(df_valid[date_col].dt.to_period('M')).size().plot(ax=ax1, marker='o')
        ax1.set_title('Articles Published Over Time', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Month', fontsize=12)
        ax1.set_ylabel('Count', fontsize=12)
        ax1.grid(True, alpha=0.3)
    except Exception as e:
        print(f"Warning: Could not create time series plot: {e}")
        ax1.text(0.5, 0.5, 'Time series plot unavailable', 
                ha='center', va='center', transform=ax1.transAxes)
    
    # Articles by class over time
    ax2 = axes[0, 1]
    try:
        df_valid.groupby([df_valid[date_col].dt.to_period('M'), 'label']).size().unstack().plot(ax=ax2)
        ax2.set_title('Articles by Class Over Time', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Month', fontsize=12)
        ax2.set_ylabel('Count', fontsize=12)
        ax2.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True, alpha=0.3)
    except Exception as e:
        print(f"Warning: Could not create class time series: {e}")
        ax2.text(0.5, 0.5, 'Class time series unavailable', 
                ha='center', va='center', transform=ax2.transAxes)
    
    # Distribution by day of week
    ax3 = axes[1, 0]
    dow_counts = df_valid['day_of_week'].value_counts().sort_index()
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    dow_counts.index = [dow_labels[int(i)] for i in dow_counts.index]
    dow_counts.plot(kind='bar', ax=ax3, color='skyblue', edgecolor='black')
    ax3.set_title('Articles by Day of Week', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Day of Week', fontsize=12)
    ax3.set_ylabel('Count', fontsize=12)
    ax3.tick_params(axis='x', rotation=45)
    
    # Distribution by month
    ax4 = axes[1, 1]
    month_counts = df_valid['month'].value_counts().sort_index()
    month_counts.plot(kind='bar', ax=ax4, color='lightcoral', edgecolor='black')
    ax4.set_title('Articles by Month', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Month', fontsize=12)
    ax4.set_ylabel('Count', fontsize=12)
    ax4.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('temporal_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n📅 Temporal Analysis Summary:")
    print("=" * 60)
    print(f"  Valid dates analyzed: {len(df_valid):,} out of {total_count:,}")
    print(f"  Date range: {df_valid[date_col].min().date()} to {df_valid[date_col].max().date()}")
    print(f"  Time span: {(df_valid[date_col].max() - df_valid[date_col].min()).days} days")
    print("=" * 60)

# Run the function
analyze_temporal_patterns(df)


In [ ]:
# Cell 11: Analyze sources and categories
def analyze_sources_categories(df: pd.DataFrame):
    """Analyze news sources and categories."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Top sources
    if 'source' in df.columns:
        ax1 = axes[0, 0]
        top_sources = df['source'].value_counts().head(15)
        top_sources.plot(kind='barh', ax=ax1, color='steelblue', edgecolor='black')
        ax1.set_title('Top 15 News Sources', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Count', fontsize=12)
        ax1.set_ylabel('Source', fontsize=12)
        ax1.invert_yaxis()
        
        # Sources by class
        ax2 = axes[0, 1]
        source_class = pd.crosstab(df['source'], df['label'])
        top_source_names = top_sources.index[:10]
        source_class.loc[top_source_names].plot(kind='bar', stacked=True, ax=ax2)
        ax2.set_title('Top 10 Sources by Class', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Source', fontsize=12)
        ax2.set_ylabel('Count', fontsize=12)
        ax2.tick_params(axis='x', rotation=45)
        ax2.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Top categories
    if 'category' in df.columns:
        ax3 = axes[1, 0]
        top_categories = df['category'].value_counts().head(15)
        top_categories.plot(kind='barh', ax=ax3, color='coral', edgecolor='black')
        ax3.set_title('Top 15 Categories', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Count', fontsize=12)
        ax3.set_ylabel('Category', fontsize=12)
        ax3.invert_yaxis()
        
        # Categories by class
        ax4 = axes[1, 1]
        cat_class = pd.crosstab(df['category'], df['label'])
        top_cat_names = top_categories.index[:10]
        cat_class.loc[top_cat_names].plot(kind='bar', stacked=True, ax=ax4)
        ax4.set_title('Top 10 Categories by Class', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Category', fontsize=12)
        ax4.set_ylabel('Count', fontsize=12)
        ax4.tick_params(axis='x', rotation=45)
        ax4.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig('source_category_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    if 'source' in df.columns:
        print("\n📰 Source Statistics:")
        print("=" * 60)
        print(f"  Total unique sources: {df['source'].nunique()}")
        print(f"\nTop 10 sources:")
        print(df['source'].value_counts().head(10))
        print("=" * 60)
    
    if 'category' in df.columns:
        print("\n📂 Category Statistics:")
        print("=" * 60)
        print(f"  Total unique categories: {df['category'].nunique()}")
        print(f"\nTop 10 categories:")
        print(df['category'].value_counts().head(10))
        print("=" * 60)

analyze_sources_categories(df)


In [ ]:
# Cell 12: Text preprocessing utilities
import unicodedata

def clean_text(text: str, preserve_bangla: bool = True) -> str:
    """
    Clean text while preserving Bangla characters.
    
    Args:
        text: Input text
        preserve_bangla: Whether to preserve Bangla Unicode characters
    
    Returns:
        Cleaned text
    """
    if pd.isna(text) or text == '':
        return ''
    
    text = str(text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Normalize Unicode (important for Bangla)
    text = unicodedata.normalize('NFKC', text)
    
    # Remove extra whitespace (but preserve Bangla spaces)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Remove zero-width characters
    text = re.sub(r'[\u200b-\u200f\ufeff]', '', text)
    
    return text

def merge_headline_description(row: pd.Series) -> str:
    """
    Merge headline and description into single text.
    
    Args:
        row: DataFrame row
    
    Returns:
        Merged and cleaned text
    """
    headline = clean_text(row.get('headline', ''))
    description = clean_text(row.get('descriptions', ''))
    
    # Combine with separator
    if headline and description:
        return f"{headline} [SEP] {description}"
    elif headline:
        return headline
    elif description:
        return description
    else:
        return ''

print("✓ Preprocessing functions defined")
print("\nPreprocessing Features:")
print("  - URL removal")
print("  - Email address removal")
print("  - Unicode normalization (NFKC for Bangla)")
print("  - Whitespace normalization")
print("  - Zero-width character removal")
print("  - Headline-description merging with [SEP] token")


In [ ]:
# Cell 13: Apply preprocessing to dataset
print("🔧 Applying preprocessing...")

# Create processed text column
df['text'] = df.apply(merge_headline_description, axis=1)

# Remove empty texts
initial_len = len(df)
df = df[df['text'].str.len() > 0].copy()
removed = initial_len - len(df)

if removed > 0:
    print(f"  ⚠️  Removed {removed} rows with empty text")

print(f"\n✓ Preprocessing complete!")
print(f"  Final dataset size: {len(df):,} rows")

# Display examples
print("\n📝 Preprocessing Examples:")
print("=" * 80)
for idx, (label, lang) in enumerate(zip(['human_true', 'human_fake'], ['bangla', 'english'])):
    sample = df[(df['label'] == label) & (df['language'] == lang)].head(1)
    if len(sample) > 0:
        print(f"\n{idx+1}. {label.upper()} ({lang.upper()}):")
        print(f"   Original headline: {sample.iloc[0].get('headline', 'N/A')[:100]}...")
        print(f"   Processed text: {sample.iloc[0]['text'][:150]}...")

print("=" * 80)

# Save processed data
df.to_csv('processed_data.csv', index=False, encoding='utf-8')
print("\n💾 Processed data saved to 'processed_data.csv'")


In [ ]:
# Cell 14: Encode labels
from sklearn.preprocessing import LabelEncoder

# Primary 4-class labels
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

# Create mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
inverse_label_mapping = {v: k for k, v in label_mapping.items()}

print("🏷️  Label Encoding:")
print("=" * 60)
for original, encoded in label_mapping.items():
    count = (df['label'] == original).sum()
    print(f"  {original:20s} → {encoded} ({count:,} samples)")
print("=" * 60)

# Save mappings for later use
CONFIG['label_mapping'] = label_mapping
CONFIG['inverse_label_mapping'] = inverse_label_mapping
CONFIG['num_labels'] = len(label_mapping)

print(f"\n✓ Total classes: {CONFIG['num_labels']}")


In [ ]:
# Cell 15: Create subsidiary binary labels for hierarchical analysis
def create_subsidiary_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create subsidiary binary labels:
    - human_vs_machine: 0=human, 1=machine
    - true_vs_fake: 0=true, 1=fake
    
    Args:
        df: Input DataFrame
    
    Returns:
        DataFrame with additional label columns
    """
    df = df.copy()
    
    # Human vs Machine
    df['is_machine'] = df['label'].apply(
        lambda x: 1 if 'machine' in str(x).lower() else 0
    )
    
    # True vs Fake
    df['is_fake'] = df['label'].apply(
        lambda x: 1 if 'fake' in str(x).lower() else 0
    )
    
    return df

df = create_subsidiary_labels(df)

print("🔀 Subsidiary Labels Created:")
print("=" * 60)

print("\n1. Human vs Machine Distribution:")
print(df['is_machine'].value_counts())
human_count = (df['is_machine'] == 0).sum()
machine_count = (df['is_machine'] == 1).sum()
print(f"   Human: {human_count:,} | Machine: {machine_count:,}")

print("\n2. True vs Fake Distribution:")
print(df['is_fake'].value_counts())
true_count = (df['is_fake'] == 0).sum()
fake_count = (df['is_fake'] == 1).sum()
print(f"   True: {true_count:,} | Fake: {fake_count:,}")

print("=" * 60)

# Cross-tabulation
print("\n📊 Cross-tabulation (Label vs Subsidiary Labels):")
cross_tab = pd.crosstab(
    df['label'], 
    [df['is_machine'], df['is_fake']], 
    rownames=['Label'], 
    colnames=['Machine', 'Fake']
)
print(cross_tab)


In [ ]:
# Cell 16: Encode language feature
language_encoder = LabelEncoder()
df['language_encoded'] = language_encoder.fit_transform(df['language'])

language_mapping = dict(zip(
    language_encoder.classes_, 
    language_encoder.transform(language_encoder.classes_)
))

print("🌐 Language Encoding:")
print("=" * 60)
for lang, code in language_mapping.items():
    count = (df['language'] == lang).sum()
    print(f"  {lang.capitalize():10s} → {code} ({count:,} samples)")
print("=" * 60)

CONFIG['language_mapping'] = language_mapping


In [ ]:
# Cell 17: Perform stratified split
from sklearn.model_selection import train_test_split

def stratified_split(df: pd.DataFrame, test_size: float = 0.2, val_size: float = 0.25, 
                     random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Perform stratified train-validation-test split.
    
    Args:
        df: Input DataFrame
        test_size: Proportion for test set
        val_size: Proportion of remaining data for validation
        random_state: Random seed
    
    Returns:
        train_df, val_df, test_df
    """
    # First split: train+val vs test
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['label_encoded'],
        random_state=random_state
    )
    
    # Second split: train vs val
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        stratify=train_val_df['label_encoded'],
        random_state=random_state
    )
    
    return train_df, val_df, test_df

# Perform split
train_df, val_df, test_df = stratified_split(
    df, 
    test_size=CONFIG['test_size'],
    val_size=CONFIG['val_size'],
    random_state=CONFIG['seed']
)

print("✂️  Data Splitting Complete!")
print("=" * 80)
print(f"\nSplit Configuration:")
print(f"  Train: {len(train_df):,} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"  Total: {len(df):,} samples")

# Verify stratification
print("\n📊 Class Distribution Verification:")
print("=" * 80)
print(f"\n{'Label':<20} {'Train':>10} {'Val':>10} {'Test':>10} {'Total':>10}")
print("-" * 80)
for label in sorted(df['label'].unique()):
    train_count = (train_df['label'] == label).sum()
    val_count = (val_df['label'] == label).sum()
    test_count = (test_df['label'] == label).sum()
    total_count = (df['label'] == label).sum()
    print(f"{label:<20} {train_count:>10,} {val_count:>10,} {test_count:>10,} {total_count:>10,}")

print("=" * 80)

# Visualize split distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (split_df, split_name) in zip(axes, [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]):
    split_df['label'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette("Set2"))
    ax.set_title(f'{split_name} Set Distribution ({len(split_df):,} samples)', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('data_split_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 18: Compute class weights for loss function
def compute_class_weights_dict(labels: np.ndarray) -> Dict[int, float]:
    """
    Compute class weights for imbalanced dataset.
    
    Args:
        labels: Array of label indices
    
    Returns:
        Dictionary mapping class index to weight
    """
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels),
        y=labels
    )
    
    weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    return weight_dict, class_weights

# Compute weights
weight_dict, class_weights_array = compute_class_weights_dict(train_df['label_encoded'].values)

print("⚖️  Class Weights (for Loss Function):")
print("=" * 60)
for label_idx, weight in weight_dict.items():
    label_name = inverse_label_mapping[label_idx]
    count = (train_df['label_encoded'] == label_idx).sum()
    print(f"  {label_name:20s} (Class {label_idx}): {weight:.4f} ({count:,} samples)")
print("=" * 60)

# Convert to tensor
class_weights_tensor = torch.FloatTensor(class_weights_array).to(CONFIG['device'])
CONFIG['class_weights'] = class_weights_tensor

print(f"\n✓ Class weights computed and saved to CONFIG")


In [ ]:
# Cell 19: SMOTE for traditional ML baseline
def apply_smote(X_train: np.ndarray, y_train: np.ndarray, random_state: int = 42):
    """
    Apply SMOTE to training data.
    
    Args:
        X_train: Training features
        y_train: Training labels
        random_state: Random seed
    
    Returns:
        Resampled X_train, y_train
    """
    print("🔄 Applying SMOTE oversampling...")
    print(f"  Original training size: {len(y_train):,}")
    
    # Check if SMOTE is applicable
    label_counts = Counter(y_train)
    min_samples = min(label_counts.values())
    
    # SMOTE requires at least 2 samples per class and k_neighbors < min_samples
    k_neighbors = min(5, min_samples - 1)
    
    if k_neighbors < 1:
        print("  ⚠️  Not enough samples for SMOTE. Skipping.")
        return X_train, y_train
    
    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    print(f"  Resampled training size: {len(y_resampled):,}")
    print(f"\n  Class distribution after SMOTE:")
    for label_idx in sorted(np.unique(y_resampled)):
        count = (y_resampled == label_idx).sum()
        label_name = inverse_label_mapping[label_idx]
        print(f"    {label_name:20s}: {count:,}")
    
    print("\n✓ SMOTE applied successfully")
    return X_resampled, y_resampled

# Note: SMOTE will be applied when training baseline models
print("SMOTE function defined and ready for baseline model training")


In [ ]:
# Cell 20: Define custom dataset class
class FakeNewsDataset(Dataset):
    """
    Custom PyTorch Dataset for fake news classification.
    """
    
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 512):
        """
        Args:
            texts: List of text strings
            labels: List of label indices
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self) -> int:
        return len(self.texts)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ FakeNewsDataset class defined")


In [ ]:
# Cell 21: Initialize XLM-RoBERTa tokenizer
print(f"📥 Loading tokenizer: {CONFIG['model_name']}")

tokenizer = XLMRobertaTokenizer.from_pretrained(CONFIG['model_name'])

print(f"\n✓ Tokenizer loaded successfully!")
print(f"  Vocabulary size: {len(tokenizer):,}")
print(f"  Special tokens: {tokenizer.all_special_tokens}")
print(f"  Max model length: {tokenizer.model_max_length:,}")

# Test tokenization on Bangla and English
sample_bangla = "এটি একটি বাংলা সংবাদ শিরোনাম।"
sample_english = "This is an English news headline."

print("\n🧪 Tokenization Test:")
print("=" * 60)
print(f"Bangla input: {sample_bangla}")
tokens_bn = tokenizer.tokenize(sample_bangla)
print(f"Tokens: {tokens_bn}")
print(f"Token IDs: {tokenizer.convert_tokens_to_ids(tokens_bn)}")

print(f"\nEnglish input: {sample_english}")
tokens_en = tokenizer.tokenize(sample_english)
print(f"Tokens: {tokens_en}")
print(f"Token IDs: {tokenizer.convert_tokens_to_ids(tokens_en)}")
print("=" * 60)


In [ ]:
# Cell 22: Create DataLoaders
def create_data_loaders(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame,
                       tokenizer, batch_size: int = 16, max_length: int = 512) -> Tuple:
    """
    Create PyTorch DataLoaders for train, validation, and test sets.
    
    Args:
        train_df, val_df, test_df: DataFrames with 'text' and 'label_encoded' columns
        tokenizer: HuggingFace tokenizer
        batch_size: Batch size
        max_length: Maximum sequence length
    
    Returns:
        train_loader, val_loader, test_loader
    """
    # Create datasets
    train_dataset = FakeNewsDataset(
        texts=train_df['text'].tolist(),
        labels=train_df['label_encoded'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )
    
    val_dataset = FakeNewsDataset(
        texts=val_df['text'].tolist(),
        labels=val_df['label_encoded'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )
    
    test_dataset = FakeNewsDataset(
        texts=test_df['text'].tolist(),
        labels=test_df['label_encoded'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    train_df, val_df, test_df,
    tokenizer=tokenizer,
    batch_size=CONFIG['batch_size'],
    max_length=CONFIG['max_length']
)

print("📦 DataLoaders Created:")
print("=" * 60)
print(f"  Train batches: {len(train_loader):,}")
print(f"  Val batches: {len(val_loader):,}")
print(f"  Test batches: {len(test_loader):,}")
print(f"  Batch size: {CONFIG['batch_size']}")
print("=" * 60)


In [ ]:
# Cell 23: Initialize XLM-RoBERTa model
print(f"🤖 Initializing XLM-RoBERTa model: {CONFIG['model_name']}")
print(f"   Number of classes: {CONFIG['num_labels']}")

model = XLMRobertaForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    problem_type="single_label_classification"
)

# Move to device
model = model.to(CONFIG['device'])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model initialized successfully!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model device: {next(model.parameters()).device}")

# Display model architecture
print(f"\n📐 Model Architecture:")
print(model)


In [ ]:
# Cell 24: Implement Focal Loss for class imbalance
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.
    
    Reference: Lin et al., "Focal Loss for Dense Object Detection" (2017)
    """
    
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        Args:
            alpha: Class weights (tensor or None)
            gamma: Focusing parameter (default: 2.0)
            reduction: 'mean' or 'sum'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Logits from model (batch_size, num_classes)
            targets: Ground truth labels (batch_size,)
        
        Returns:
            Focal loss value
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        # Apply alpha weights
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Option to use Focal Loss or weighted CrossEntropy
USE_FOCAL_LOSS = False  # Set to True to use Focal Loss

if USE_FOCAL_LOSS:
    criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)
    print("✓ Using Focal Loss with gamma=2.0")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    print("✓ Using Weighted Cross-Entropy Loss")

print(f"  Class weights: {class_weights_tensor.cpu().numpy()}")


In [ ]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Calculate total training steps
num_training_steps = len(train_loader) * CONFIG['epochs']
num_warmup_steps = int(num_training_steps * CONFIG['warmup_ratio'])

# Learning rate scheduler (CORRECTED)
# ✅ CORRECTED: Use get_scheduler from transformers
scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)


print("⚙️  Training Configuration:")
print("=" * 60)
print(f"  Optimizer: AdamW")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Weight decay: {CONFIG['weight_decay']}")
print(f"  Scheduler: Linear with warmup")
print(f"  Warmup steps: {num_warmup_steps:,} ({CONFIG['warmup_ratio']*100:.0f}%)")
print(f"  Total steps: {num_training_steps:,}")
print(f"  Epochs: {CONFIG['epochs']}")
print("=" * 60)


In [ ]:
# Cell 26: Enhanced Baseline Linear SVM with Hyperparameter Tuning and Visualizations

print("🔧 Training Enhanced Baseline: Linear SVM with TF-IDF")
print("=" * 60)

# 1. Enhanced TF-IDF Vectorization
print("\n1. Enhanced TF-IDF Vectorization...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,          # ✅ Increased from 10000
    ngram_range=(1, 2),          # Keep bi-grams
    min_df=2,                     # Filter rare words
    max_df=0.95,                  # Filter very common words
    strip_accents='unicode',
    sublinear_tf=True            # ✅ Added: Use sublinear TF scaling (1 + log(tf))
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['text'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['text'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['text'])

print(f"  Train shape: {X_train_tfidf.shape}")
print(f"  Val shape: {X_val_tfidf.shape}")
print(f"  Test shape: {X_test_tfidf.shape}")

# Extract labels
y_train_baseline = train_df['label_encoded'].values
y_val_baseline = val_df['label_encoded'].values
y_test_baseline = test_df['label_encoded'].values

# 2. Apply SMOTE
APPLY_SMOTE = True
if APPLY_SMOTE:
    print("\n2. Applying SMOTE...")
    X_train_resampled, y_train_resampled = apply_smote(
        X_train_tfidf.toarray(), 
        y_train_baseline,
        random_state=CONFIG['seed']
    )
else:
    X_train_resampled = X_train_tfidf
    y_train_resampled = y_train_baseline

# 3. Hyperparameter Tuning with GridSearchCV
print("\n3. Hyperparameter Tuning...")
print("🔍 Searching for best SVM parameters...")

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1.0, 10.0],                    # ✅ Regularization parameter
    'class_weight': ['balanced', None],       # ✅ Test with/without balancing
    'loss': ['hinge', 'squared_hinge']        # ✅ Test different loss functions
}

svm_base = LinearSVC(
    random_state=CONFIG['seed'],
    max_iter=5000,
    dual=False
)

grid_search = GridSearchCV(
    svm_base,
    param_grid,
    cv=3,                    # 3-fold cross-validation
    scoring='f1_macro',      # Optimize for macro F1
    n_jobs=-1,               # Use all CPU cores
    verbose=1
)

grid_search.fit(X_train_resampled, y_train_resampled)

print(f"\n  ✓ Best parameters: {grid_search.best_params_}")
print(f"  ✓ Best CV F1-score: {grid_search.best_score_:.4f}")

# Use the best model
svm_model = grid_search.best_estimator_

# 4. Predictions
print("\n4. Making predictions...")
y_val_pred_svm = svm_model.predict(X_val_tfidf)
y_test_pred_svm = svm_model.predict(X_test_tfidf)

# 5. Evaluation
val_acc_svm = accuracy_score(y_val_baseline, y_val_pred_svm)
test_acc_svm = accuracy_score(y_test_baseline, y_test_pred_svm)

print(f"\n✓ SVM Training Complete!")
print(f"  Validation Accuracy: {val_acc_svm:.4f}")
print(f"  Test Accuracy: {test_acc_svm:.4f}")

# 6. Detailed Classification Report
print("\n📊 SVM Classification Report (Test Set):")
print("=" * 60)
class_names = [inverse_label_mapping[i] for i in range(CONFIG['num_labels'])]
print(classification_report(
    y_test_baseline, 
    y_test_pred_svm,
    target_names=class_names,
    digits=4
))
print("=" * 60)

# 7. Generate Confusion Matrices
print("\n📊 Generating Confusion Matrices...")

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Compute confusion matrix
cm = confusion_matrix(y_test_baseline, y_test_pred_svm)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('SVM Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# Normalized confusion matrix
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('SVM Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrices saved: svm_confusion_matrix.png")

# 8. Per-Class Metrics
print("\n📈 SVM Per-Class Metrics:")
print("=" * 60)

from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_baseline, y_test_pred_svm, average=None
)

print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 60)
for i, class_name in enumerate(class_names):
    print(f"{class_name:<20} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10,}")

macro_f1 = f1.mean()
weighted_f1 = np.average(f1, weights=support)
print("-" * 60)
print(f"{'Macro Avg':<20} {precision.mean():>10.4f} {recall.mean():>10.4f} {macro_f1:>10.4f}")
print(f"{'Weighted Avg':<20} {np.average(precision, weights=support):>10.4f} {np.average(recall, weights=support):>10.4f} {weighted_f1:>10.4f}")
print("=" * 60)

# 9. ROC Curves
print("\n📉 Generating ROC Curves...")

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, roc_auc_score

# Binarize labels for multi-class ROC
y_test_bin = label_binarize(y_test_baseline, classes=range(CONFIG['num_labels']))

# Get decision scores
decision_scores = svm_model.decision_function(X_test_tfidf)

# Compute ROC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(CONFIG['num_labels']):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], decision_scores[:, i])
    roc_auc[i] = roc_auc_score(y_test_bin[:, i], decision_scores[:, i])

# Plot ROC curves
plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'orange']

for i, color in zip(range(CONFIG['num_labels']), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"{class_names[i]} (AUC = {roc_auc[i]:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('SVM ROC Curves (Multi-class)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('svm_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ ROC curves saved: svm_roc_curves.png")

# Compute micro and macro AUC
micro_auc = roc_auc_score(y_test_bin, decision_scores, average='micro')
macro_auc = roc_auc_score(y_test_bin, decision_scores, average='macro')

print(f"\n🎯 SVM ROC-AUC Scores:")
print("=" * 60)
for i, class_name in enumerate(class_names):
    print(f"  {class_name:<20}: {roc_auc[i]:.4f}")
print("-" * 60)
print(f"  Micro-average AUC: {micro_auc:.4f}")
print(f"  Macro-average AUC: {macro_auc:.4f}")
print("=" * 60)

# 10. Error Analysis
print("\n🔍 SVM Error Analysis:")
print("=" * 60)

misclassified_mask = y_test_pred_svm != y_test_baseline
print(f"  Total test samples: {len(y_test_baseline):,}")
print(f"  Correct predictions: {(~misclassified_mask).sum():,} ({(~misclassified_mask).mean()*100:.2f}%)")
print(f"  Incorrect predictions: {misclassified_mask.sum():,} ({misclassified_mask.mean()*100:.2f}%)")
print("=" * 60)

print("\n📊 Errors by True Label:")
print("-" * 60)
for i, class_name in enumerate(class_names):
    class_mask = y_test_baseline == i
    class_errors = misclassified_mask & class_mask
    total_class = class_mask.sum()
    error_count = class_errors.sum()
    error_rate = (error_count / total_class * 100) if total_class > 0 else 0
    print(f"  {class_name:20s} {error_count:5,} / {total_class:5,}  ({error_rate:5.2f}%)")
print("-" * 60)

# 11. Save baseline results (enhanced)
baseline_results = {
    'model': 'Linear SVM (Enhanced)',
    'val_accuracy': float(val_acc_svm),
    'test_accuracy': float(test_acc_svm),
    'macro_f1': float(macro_f1),
    'weighted_f1': float(weighted_f1),
    'macro_auc': float(macro_auc),
    'micro_auc': float(micro_auc),
    'best_params': grid_search.best_params_,
    'cv_best_score': float(grid_search.best_score_),
    'predictions': y_test_pred_svm
}

print("\n✅ Enhanced SVM Training and Evaluation Complete!")
print(f"   Confusion matrices saved: svm_confusion_matrix.png")
print(f"   ROC curves saved: svm_roc_curves.png")


In [ ]:
# Cell 27: Define training and evaluation functions
def train_epoch(model, data_loader, criterion, optimizer, scheduler, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(data_loader, desc="Training", leave=False)
    
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Calculate loss
        loss = criterion(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * correct / total:.2f}%'
        })
    
    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy

def evaluate(model, data_loader, criterion, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    all_logits = []
    
    progress_bar = tqdm(data_loader, desc="Evaluating", leave=False)
    
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Calculate loss
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Store predictions
            _, predicted = torch.max(logits, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_logits.append(logits.cpu().numpy())
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    all_logits = np.vstack(all_logits)
    
    return avg_loss, accuracy, all_predictions, all_labels, all_logits

print("✓ Training and evaluation functions defined")


In [ ]:
# Cell 28: Train XLM-RoBERTa model
print("🚀 Starting XLM-RoBERTa Training")
print("=" * 80)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_state = None
epochs_no_improve = 0

for epoch in range(CONFIG['epochs']):
    print(f"\n📅 Epoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 80)
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, CONFIG['device']
    )
    
    # Validate
    val_loss, val_acc, _, _, _ = evaluate(
        model, val_loader, criterion, CONFIG['device']
    )
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print epoch results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Check for improvement
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        epochs_no_improve = 0
        print(f"  ✓ New best validation accuracy: {best_val_acc:.4f}")
        
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss
        }, 'best_model_checkpoint.pt')
        print(f"  💾 Checkpoint saved")
    else:
        epochs_no_improve += 1
        print(f"  ⏸  No improvement for {epochs_no_improve} epoch(s)")
    
    # Early stopping
    if epochs_no_improve >= CONFIG['early_stopping_patience']:
        print(f"\n🛑 Early stopping triggered after {epoch + 1} epochs")
        break
    
    # Memory cleanup
    torch.cuda.empty_cache()
    gc.collect()

# Load best model
print(f"\n✓ Training complete!")
print(f"  Best validation accuracy: {best_val_acc:.4f}")
model.load_state_dict(best_model_state)
print("  ✓ Best model loaded")

print("\n" + "=" * 80)


In [ ]:
# Cell 29: Plot training curves
def plot_training_history(history: Dict):
    """Plot training and validation metrics over epochs."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # Loss plot
    ax1 = axes[0]
    ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
    ax1.plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss', linewidth=2)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2 = axes[1]
    ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
    ax2.plot(epochs_range, history['val_acc'], 'r-s', label='Val Acc', linewidth=2)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)


In [ ]:
# Cell 30: Evaluate on test set
print("🧪 Evaluating on Test Set")
print("=" * 80)

test_loss, test_acc, test_predictions, test_labels, test_logits = evaluate(
    model, test_loader, criterion, CONFIG['device']
)

print(f"\n📊 Test Set Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_predictions, average=None
)

print(f"\n📈 Per-Class Metrics:")
print("=" * 80)
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 80)
for i in range(CONFIG['num_labels']):
    label_name = inverse_label_mapping[i]
    print(f"{label_name:<20} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10,}")

# Macro and weighted averages
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_labels, test_predictions, average='macro'
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    test_labels, test_predictions, average='weighted'
)

print("-" * 80)
print(f"{'Macro Avg':<20} {macro_precision:>10.4f} {macro_recall:>10.4f} {macro_f1:>10.4f}")
print(f"{'Weighted Avg':<20} {weighted_precision:>10.4f} {weighted_recall:>10.4f} {weighted_f1:>10.4f}")
print("=" * 80)

# Classification report
print(f"\n📋 Detailed Classification Report:")
print("=" * 80)
print(classification_report(
    test_labels, 
    test_predictions,
    target_names=[inverse_label_mapping[i] for i in range(CONFIG['num_labels'])],
    digits=4
))
print("=" * 80)


In [ ]:
# Cell 31: Plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix'):
    """Plot confusion matrix with annotations."""
    
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot heatmap
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'},
        ax=ax,
        linewidths=0.5,
        linecolor='gray'
    )
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Normalized confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        cm_normalized, 
        annot=True, 
        fmt='.2%', 
        cmap='Greens',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Percentage'},
        ax=ax,
        linewidths=0.5,
        linecolor='gray'
    )
    
    ax.set_title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
    plt.show()

class_names = [inverse_label_mapping[i] for i in range(CONFIG['num_labels'])]
plot_confusion_matrix(test_labels, test_predictions, class_names)


In [ ]:
# Cell 32: ROC-AUC curves (One-vs-Rest)
from sklearn.preprocessing import label_binarize

# Binarize labels
y_test_bin = label_binarize(test_labels, classes=list(range(CONFIG['num_labels'])))

# Compute probabilities
test_probs = F.softmax(torch.tensor(test_logits), dim=1).numpy()

# Plot ROC curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i in range(CONFIG['num_labels']):
    # Compute ROC curve
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], test_probs[:, i])
    roc_auc = roc_auc_score(y_test_bin[:, i], test_probs[:, i])
    
    # Plot
    ax = axes[i]
    ax.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}', linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.set_title(f'ROC Curve: {inverse_label_mapping[i]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# Overall micro and macro AUC
micro_auc = roc_auc_score(y_test_bin, test_probs, average='micro')
macro_auc = roc_auc_score(y_test_bin, test_probs, average='macro')

print(f"\n🎯 Overall ROC-AUC Scores:")
print("=" * 60)
print(f"  Micro-average AUC: {micro_auc:.4f}")
print(f"  Macro-average AUC: {macro_auc:.4f}")
print("=" * 60)


In [ ]:
# Cell 33: Perform error analysis
def error_analysis(df_test: pd.DataFrame, predictions: np.ndarray, true_labels: np.ndarray):
    """Analyze misclassified examples."""
    
    df_test = df_test.copy()
    df_test['predicted'] = [inverse_label_mapping[p] for p in predictions]
    df_test['true_label'] = [inverse_label_mapping[t] for t in true_labels]
    df_test['correct'] = df_test['predicted'] == df_test['true_label']
    
    # Count errors
    errors = df_test[~df_test['correct']]
    print(f"🔍 Error Analysis:")
    print("=" * 80)
    print(f"  Total test samples: {len(df_test):,}")
    print(f"  Correct predictions: {df_test['correct'].sum():,} ({df_test['correct'].mean()*100:.2f}%)")
    print(f"  Incorrect predictions: {len(errors):,} ({(1-df_test['correct'].mean())*100:.2f}%)")
    print("=" * 80)
    
    # Error breakdown by true label
    print(f"\n📊 Errors by True Label:")
    print("-" * 80)
    for label in df_test['true_label'].unique():
        label_df = df_test[df_test['true_label'] == label]
        label_errors = len(label_df[~label_df['correct']])
        error_rate = label_errors / len(label_df) * 100
        print(f"  {label:<20} {label_errors:>5} / {len(label_df):<5} ({error_rate:>5.2f}%)")
    print("-" * 80)
    
    # Common misclassification pairs
    print(f"\n🔄 Most Common Misclassifications:")
    print("-" * 80)
    misclass_pairs = errors.groupby(['true_label', 'predicted']).size().sort_values(ascending=False)
    for (true_lbl, pred_lbl), count in misclass_pairs.head(10).items():
        print(f"  {true_lbl:<20} → {pred_lbl:<20} ({count:>3} cases)")
    print("-" * 80)
    
    # Sample misclassified examples
    print(f"\n📝 Sample Misclassified Examples:")
    print("=" * 80)
    for i, (true_lbl, pred_lbl) in enumerate(misclass_pairs.head(3).index):
        sample = errors[(errors['true_label'] == true_lbl) & 
                       (errors['predicted'] == pred_lbl)].iloc[0]
        print(f"\n{i+1}. True: {true_lbl} | Predicted: {pred_lbl}")
        print(f"   Language: {sample.get('language', 'N/A')}")
        print(f"   Text: {sample['text'][:200]}...")
        print("-" * 80)
    
    return df_test

df_test_analyzed = error_analysis(test_df, test_predictions, test_labels)


In [ ]:
# Cell 34: Analyze performance by language
def analyze_by_language(df_test: pd.DataFrame):
    """Analyze model performance by language."""
    
    print(f"\n🌍 Performance by Language:")
    print("=" * 80)
    
    for lang in df_test['language'].unique():
        lang_df = df_test[df_test['language'] == lang]
        if len(lang_df) == 0:
            continue
        
        accuracy = lang_df['correct'].mean()
        print(f"\n{lang.capitalize()}:")
        print(f"  Samples: {len(lang_df):,}")
        print(f"  Accuracy: {accuracy:.4f}")
        
        # Per-class accuracy
        print(f"  Per-class accuracy:")
        for label in lang_df['true_label'].unique():
            label_df = lang_df[lang_df['true_label'] == label]
            if len(label_df) > 0:
                label_acc = label_df['correct'].mean()
                print(f"    {label:<20}: {label_acc:.4f} ({len(label_df):>4} samples)")
    
    print("=" * 80)

analyze_by_language(df_test_analyzed)


In [ ]:
# Cell 35: Compare XLM-RoBERTa with SVM baseline
print(f"\n⚖️  Model Comparison:")
print("=" * 80)

comparison = pd.DataFrame({
    'Model': ['Linear SVM (TF-IDF)', 'XLM-RoBERTa'],
    'Validation Accuracy': [baseline_results['val_accuracy'], best_val_acc],
    'Test Accuracy': [baseline_results['test_accuracy'], test_acc],
    'Test Macro F1': [
        f1_score(y_test_baseline, baseline_results['predictions'], average='macro'),
        macro_f1
    ],
    'Test Weighted F1': [
        f1_score(y_test_baseline, baseline_results['predictions'], average='weighted'),
        weighted_f1
    ]
})

print(comparison.to_string(index=False))
print("=" * 80)

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison))
width = 0.35

metrics = ['Validation Accuracy', 'Test Accuracy', 'Test Macro F1', 'Test Weighted F1']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, metric in enumerate(metrics):
    values = comparison[metric].values
    ax.bar(x + i*width/len(metrics), values, width/len(metrics), 
           label=metric, color=colors[i], alpha=0.8, edgecolor='black')

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width/2)
ax.set_xticklabels(comparison['Model'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import json
import numpy as np

def convert_to_serializable(obj):
    """Convert numpy/pandas types to Python types for JSON."""
    if isinstance(obj, dict):
        return {str(k): convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif pd.isna(obj):
        return None
    else:
        return obj


In [ ]:
# Cell 36: Save results summary (FINAL - MINIMAL VERSION)

import json
import numpy as np
import torch

def convert_to_serializable(obj):
    """Convert numpy/pandas/torch types to Python types for JSON."""
    if isinstance(obj, dict):
        return {str(k): convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_to_serializable(item) for item in obj)
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy().tolist()
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif pd.isna(obj):
        return None
    else:
        return obj

print("💾 Saving results summary...")

# Convert to serializable format
results_summary_clean = convert_to_serializable(results_summary)

# Save to JSON
with open('results_summary.json', 'w', encoding='utf-8') as f:
    json.dump(results_summary_clean, f, indent=4, ensure_ascii=False)

print("✅ Results summary saved to 'results_summary.json'")
print(f"   Saved {len(results_summary_clean)} items to JSON file")


In [ ]:
# Cell 37: Generate final summary report
print("\n" + "="*80)
print("🎉 FINAL SUMMARY REPORT")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"  Total samples: {len(df):,}")
print(f"  Training samples: {len(train_df):,}")
print(f"  Validation samples: {len(val_df):,}")
print(f"  Test samples: {len(test_df):,}")
print(f"  Number of classes: {CONFIG['num_labels']}")
print(f"  Languages: Bangla, English, Mixed")

print(f"\n🤖 Model Architecture:")
print(f"  Base model: {CONFIG['model_name']}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Max sequence length: {CONFIG['max_length']}")

print(f"\n⚙️  Training Configuration:")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Epochs trained: {len(history['train_loss'])}")
print(f"  Early stopping patience: {CONFIG['early_stopping_patience']}")
print(f"  Class imbalance handling: Weighted loss + SMOTE (baseline)")

print(f"\n📈 Best Results:")
print(f"  Validation Accuracy: {best_val_acc:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test Macro F1: {macro_f1:.4f}")
print(f"  Test Weighted F1: {weighted_f1:.4f}")
print(f"  Macro AUC: {macro_auc:.4f}")

print(f"\n🎯 Per-Class Performance (Test Set):")
for i in range(CONFIG['num_labels']):
    label_name = inverse_label_mapping[i]
    print(f"  {label_name}:")
    print(f"    Precision: {precision[i]:.4f}")
    print(f"    Recall: {recall[i]:.4f}")
    print(f"    F1-Score: {f1[i]:.4f}")

print(f"\n🔬 Key Insights:")
print(f"  1. XLM-RoBERTa outperforms Linear SVM baseline")
print(f"  2. Class imbalance handled via weighted loss")
print(f"  3. Early stopping prevents overfitting")
print(f"  4. Model performs well on both Bangla and English")
print(f"  5. Four-class detection more challenging than binary")

print(f"\n💡 Recommendations:")
print(f"  • Consider two-stage (conjoint) approach: human/machine → true/fake")
print(f"  • Experiment with larger models (XLM-RoBERTa-large)")
print(f"  • Augment machine-generated samples for minority classes")
print(f"  • Apply ensemble methods combining multiple models")
print(f"  • Fine-tune hyperparameters via grid/random search")

print("\n" + "="*80)
print("✅ PIPELINE EXECUTION COMPLETE!")
print("="*80)

print("\n📁 Output Files Generated:")
print("  ✓ processed_data.csv")
print("  ✓ best_model_checkpoint.pt")
print("  ✓ xlm_roberta_fakenews_model/")
print("  ✓ results_summary.json")
print("  ✓ test_predictions.csv")
print("  ✓ All visualization plots (PNG)")

print("\n🚀 Next Steps:")
print("  1. Deploy model for real-time inference")
print("  2. Build interactive dashboard for predictions")
print("  3. Conduct user study for practical validation")
print("  4. Extend to other low-resource languages")
print("  5. Integrate with fact-checking pipeline")

print("\n" + "="*80)


In [ ]:
import shutil

# Compress everything in /kaggle/working into a single zip file
shutil.make_archive('/kaggle/working/all_files', 'zip', '/kaggle/working')
